In [3]:
import sys
!{sys.executable} -m pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import os
import ast
import pandas as pd
import json
import numpy as np
from neo4j import GraphDatabase

In [5]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

In [6]:
uri = "bolt://neo4j-gds-apoc-n10s:7687"
username = "neo4j"
password = "neo4jpassword"

In [7]:
# Jupyter docker내에서 억세스 할 수 있는 데이터 위치
HOME_DIR = "/app/src/app/data/data/rdb_reformulate/"

### neo4j database 초기화
- 다른 필요한 데이터가 들어있을 경우 실행하면 안 됨

In [8]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
   # 모든 관계와 노드 삭제
   session.run("MATCH (n) DETACH DELETE n")

driver.close()

### ProductMeta 노드 생성 (root 노드)
- Label: DatabaseType
- Property
  - type: 'ProductMeta'

In [9]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    session.run("CREATE (:DB타입 {타입: '상품메타'})")

driver.close()

### 상위 카테고리 노드 생성
- Label: Category
- Property
  - name: 상위 category 이름
- Parent Relation
  - DatabaseType('ProductMeta') -[ HAS_category ]-> Category('...')

In [10]:
name_list = ['AutoProductChange', 'Benefit', 'Campaign', 'CommonRule', 'CustomerInfo', 'Data', 'OptionData', 'Product', 'SmsText', 'TopupInfo', 'Voice']
# name_list = [ "Benefit", "Product" ]
kor_name_list = ["요금제자동변경", "리필혜택", "혜택", "공통규칙", "가입조건", "기본제공데이터", "옵션데이터", "상품", "기본제공문자", "충전", "기본제공음성"]

driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    # Category 노드 생성
    # for name in name_list:
    #     session.run(f"CREATE (:Category {{name: '{name}'}})")
    for name in kor_name_list:
        session.run(f"CREATE (:카테고리 {{이름: '{name}'}})")

    # ProductMeta 노드와 Category 노드 연결
    # session.run("""
    #    MATCH (pm:DatabaseType {type: 'ProductMeta'})
    #    MATCH (c:Category)
    #    CREATE (pm)-[:HAS_Category]->(c)
    # """)
    session.run("""
       MATCH (pm:DB타입 {타입: '상품메타'})
       MATCH (c:카테고리)
       CREATE (pm)-[:상품메타_종류]->(c)
    """)

driver.close()

### product > plan
- 모바일 요금제 상품
- Label: Plan
- Property
  - 요금제 속성들
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...')

In [11]:
meta_path = os.path.join(HOME_DIR, "product", "plan", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [12]:
sub_meta_path = os.path.join(HOME_DIR, "product", "meta.csv")
sub_meta_table = pd.read_csv(sub_meta_path)

In [13]:
meta_table = meta_table.merge(sub_meta_table, on=["pmProductId"])

In [14]:
meta_table.head(5)

,pmProductId,approvalinfo,classifiedgroup,generation,lineup,marketingkeyword,productdescription,productid,productnameinenglish,productoperationperiod,productrequirednotice,productsubscriptioncondition,productsubscriptionmethod,statusofoperation,versioninfo,productoperationperiodFrom,productoperationperiodTo,billingmethod,monthlypricewithoutvat,monthlypricewithselectableinstallment,monthlyprice,productName,legacyProductId,type
0,PA00000001,요금팀/이현정,상품 > 기본요금제 > 휴대폰 요금제,"['LTE', '5G']",다이렉트플랜,"['유심개통', '쓰던폰', 'USIM개통', '비대면', '자급제', '온라인', '데이터100GB 이하', 'T다샵', '다이렉트플랜', '티다이렉트샵', '티다샵', 'wavve할인혜택', '온라인전용요금제', 'T다이렉트전용요금제']",월 15GB 데이터를 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제,PA00000001,Direct5G 38,2021.01.14~9999.12.31,NaN,T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능,"고객센터, 대리점, T월드 가입 가능",False,1.0,2021-01-14,2262-04-11,후불,34546,38000,38000,다이렉트5G 38,NA00007164,plan
1,PA00000002,요금팀/이현정,상품 > 기본요금제 > 휴대폰 요금제,"['LTE', '5G']",LTE 다이렉트플랜,"['자급제', '비대면', '온라인', '티다샵', '티다이렉트샵', '유심개통', 'USIM개통', '쓰던폰', '심야데이터할인', 'T다이렉트전용요금제', '온라인전용요금제', '다이렉트플랜', 'T다샵']",월 1.8GB 데이터를 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제,PA00000002,DirectLTE 22,2021.01.14~9999.12.31,NaN,T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능,"고객센터, 대리점, T월드 가입 가능",False,1.0,2021-01-14,2262-04-11,후불,20000,22000,22000,다이렉트LTE 22,NA00007166,plan
2,PA00000003,요금팀/이현정,상품 > 기본요금제 > 휴대폰 요금제,"['LTE', '5G']",LTE 다이렉트플랜,"['쓰던폰', '자급제', '비대면', '온라인', '티다이렉트샵', '유심개통', 'USIM개통', '심야데이터할인', 'T다이렉트전용요금제', '온라인전용요금제', '다이렉트플랜', 'T다샵', '티다샵']",월 5GB 데이터를 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제,PA00000003,DirectLTE 35,2021.01.14~9999.12.31,NaN,T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능,"고객센터, 대리점, T월드 가입 가능",False,1.0,2021-01-14,2262-04-11,후불,31819,35000,35000,다이렉트LTE 35,NA00007167,plan
3,PA00000004,요금팀/이현정,상품 > 기본요금제 > 휴대폰 요금제,"['LTE', '5G']",LTE 다이렉트플랜,"['온라인', '유심개통', 'USIM개통', '쓰던폰', '자급제', '100GB이상', '비대면', '티다샵', '티다이렉트샵', 'T다이렉트전용요금제', '온라인전용요금제', '다이렉트플랜', 'T다샵']",월 100GB 데이터를 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제,PA00000004,DirectLTE 48,2021.01.14~9999.12.31,NaN,T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능,"고객센터, 대리점, T월드 가입 가능",False,1.0,2021-01-14,2262-04-11,후불,43637,48000,48000,다이렉트LTE 48,NA00007168,plan
4,PA00000005,요금팀/이현정,상품 > 기본요금제 > 휴대폰 요금제,"['LTE', '5G']",다이렉트플랜,"['온라인', '티다이렉트샵', '티다샵', 'USIM개통', '유심개통', '자급제', '쓰던폰', '비대면', '100GB이상', 'wavve할인혜택', '온라인전용요금제', 'T다이렉트전용요금제', 'T다샵', '다이렉트플랜']",월 200GB 데이터를 제공하면서 wavve 혜택도 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제,PA00000005,Direct5G 52,2021.01.14~9999.12.31,NaN,T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능,"고객센터, 대리점, T월드 가입 가능",False,1.0,2021-01-14,2262-04-11,후불,47273,52000,52000,다이렉트5G 52,NA00007183,plan


In [15]:
cols_list = [
    'pmProductId', 'generation',
    'lineup', 'productdescription', 'productid',
    'productnameinenglish',
    'productsubscriptioncondition',
    'productsubscriptionmethod', 
    'productoperationperiodFrom', 'productoperationperiodTo',
    'monthlypricewithoutvat',
    'monthlypricewithselectableinstallment', 'monthlyprice',
    'productName', 'legacyProductId'
]
cols = set(cols_list)

In [16]:
kor_cols_list = [
    "고유ID", "통신규격", 
    "라인업", "상품설명", "상품ID", 
    "상품영문명", 
    "상품가입조건", 
    "관리정보", 
    "상품운영시작일", "상품운영종료일", 
    "VAT제외월정액", 
    "선택약정VAT포함월정액", "VAT포함월정액",
    "상품명", "과거상품ID"
]
# kor_cols = set(kor_col_list)

In [17]:
kor_cols_map = {x:y for x, y in zip(cols_list, kor_cols_list)}

In [18]:
def type_cast(input_data):
    new_data = None
    if '[' in input_data and ']' in input_data and 'nan' in input_data:
        input_data = input_data.replace("nan", "")
    new_data = ast.literal_eval(input_data)

    return new_data

In [19]:
# 리스트형으로 변환
meta_table["generation"] = meta_table["generation"].apply(lambda x: type_cast(x))
meta_table["marketingkeyword"] = meta_table["marketingkeyword"].apply(lambda x: type_cast(x))

In [20]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for _, row in meta_table.iterrows():
        props = {}
        for col, val in row.items():
            if col not in cols:
                continue
            if col in ('generation', 'marketingkeyword'):
                # props[col] = list(val) if isinstance(val, (list, tuple)) else []
                props[kor_cols_map[col]] = list(val) if isinstance(val, (list, tuple)) else []
            else:
                # NaN/None 스킵
                if pd.isna(val):
                    continue
                # pandas dtype → Python 기본 타입
                if isinstance(val, (np.integer, int)):
                    # props[col] = int(val)
                    props[kor_cols_map[col]] = int(val)
                elif isinstance(val, (np.floating, float)):
                    # props[col] = float(val)
                    props[kor_cols_map[col]] = float(val)
                elif isinstance(val, (np.bool_, bool)):
                    # props[col] = bool(val)
                    props[kor_cols_map[col]] = bool(val)
                else:
                    # props[col] = str(val)
                    props[kor_cols_map[col]] = str(val)

        session.run(
            # "CREATE (n:Plan $props)",
            "CREATE (n:요금제 $props)",
            props=props
        )
driver.close()

In [21]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    # session.run("""
    #     MATCH (c:Category {name: 'Product'})
    #     MATCH (pl:Plan)
    #     CREATE (c)-[:HAS_Plan]->(pl)
    # """)
    session.run("""
        MATCH (c:카테고리 {이름: '상품'})
        MATCH (pl:요금제)
        CREATE (c)-[:요금제상품]->(pl)
    """)

driver.close()

### product > plan > marketingKeyword
- 마케팅키워드를 분리해서 노드로 변경

In [22]:
meta_table[["pmProductId", "marketingkeyword"]]

,pmProductId,marketingkeyword
0,PA00000001,"[유심개통, 쓰던폰, USIM개통, 비대면, 자급제, 온라인, 데이터100GB 이하, T다샵, 다이렉트플랜, 티다이렉트샵, 티다샵, wavve할인혜택, 온라인전용요금제, T다이렉트전용요금제]"
1,PA00000002,"[자급제, 비대면, 온라인, 티다샵, 티다이렉트샵, 유심개통, USIM개통, 쓰던폰, 심야데이터할인, T다이렉트전용요금제, 온라인전용요금제, 다이렉트플랜, T다샵]"
2,PA00000003,"[쓰던폰, 자급제, 비대면, 온라인, 티다이렉트샵, 유심개통, USIM개통, 심야데이터할인, T다이렉트전용요금제, 온라인전용요금제, 다이렉트플랜, T다샵, 티다샵]"
3,PA00000004,"[온라인, 유심개통, USIM개통, 쓰던폰, 자급제, 100GB이상, 비대면, 티다샵, 티다이렉트샵, T다이렉트전용요금제, 온라인전용요금제, 다이렉트플랜, T다샵]"
4,PA00000005,"[온라인, 티다이렉트샵, 티다샵, USIM개통, 유심개통, 자급제, 쓰던폰, 비대면, 100GB이상, wavve할인혜택, 온라인전용요금제, T다이렉트전용요금제, T다샵, 다이렉트플랜]"
...,...,...
120,PA00002811,"[Wavve, 로밍할인, 스마트기기, 커피, 갤럭시워치, 커피50%, 혜택요금제, 스트리밍, 영화50%, 애플워치, 영화, baro50%, 커피할인, 커피쿠폰, 데이터무제한, 영화쿠폰, 로밍쿠폰, 로밍, wavve혜택, 영화할인, 만34세이하]"
121,PA00002812,"[로밍할인, 데이터무제한, 커피할인, 영화50%, 유튜브요금제, 유튜브혜택, 영화, 커피쿠폰, Wavve, 커피, wavve혜택, baro50%, 영화쿠폰, YoutubePremium혜택, 혜택요금제, 로밍, 스트리밍, 로밍쿠폰, 영화할인, 만34세이하, 유튜브무료제공요금제, 커피50%]"
122,PA00002813,"[쓰던폰, 유심개통, 태블릿기기할인, 자급제, 온라인전용요금제, 티다이렉트샵, Wavve, 스트리밍, 티다샵, VIP멤버십혜택, T다이렉트전용요금제, 스마트기기, 태블릿요금무료혜택, T다샵, 비대면, 혜택요금제, 애플워치, 스마트워치요금무료혜택, 스마트폰기기할인, 데이터무제한, 갤럭시워치, 스마트워치기기할인, USIM개통, 온라인, wavve혜택]"
123,PA00002814,"[VIP멤버십혜택, 자급제, 유심개통, 우주패스, T다샵, 쓰던폰, 디즈니혜택, 태블릿요금무료혜택, 티다샵, 스트리밍, 스마트워치요금무료혜택, 혜택요금제, T다이렉트전용요금제, 디즈니, 비대면, 티다이렉트샵, 디즈니플러스, USIM개통, 온라인전용요금제, OTT혜택, Disney, 데이터무제한, 온라인, wavve혜택, Wavve]"


In [23]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for _, row in meta_table[["pmProductId", "marketingkeyword"]].iterrows():
        pmProductId = row["pmProductId"]
        for keyword in row["marketingkeyword"]:
            cypher = f"""
            MERGE (p:요금제 {{고유ID: '{pmProductId}'}})
            MERGE (k:마케팅키워드 {{값: '{keyword}'}})
            MERGE (p)-[:마케팅키워드]->(k)
            """
            session.run(cypher)

driver.close()

### autoProductChange
- 요금제 자동 변경 정책
- Label: X (relation only)
- Property
  - 변경 조건들 (changerule_base, changerule_date)
- Parent Relation
  - Product('...') -[relationType 칼럼 값]-> Product('...')

In [24]:
meta_path = os.path.join(HOME_DIR, "autoProductChange", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [25]:
meta_table = meta_table.rename(columns={
    "autoproductchange|changerule|datebase|value": "changerule_base",
    "autoproductchange|changerule|date|value": "changerule_date"
})

In [26]:
meta_table["relationType"].unique()

array(['autoproductchange'], dtype=object)

In [27]:
kor_relation_type_map = { "autoproductchange": "요금제자동변경" }

In [28]:
meta_table["changerule_base"].unique()

array(['만36세', '만14세', '제대일', '만20세', '넷플릭스 가입일'], dtype=object)

In [29]:
meta_table["changerule_date"].unique()

array(['기준일 + 1개월 1일', '기준일 + 8일'], dtype=object)

In [30]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for _, row in meta_table.iterrows():
        src = row["pmProductId"]
        tgt = row["targetProductId"]
        base = row["changerule_base"]
        date = row["changerule_date"]
        # rel_type = row["relationType"]
        rel_type = kor_relation_type_map[row["relationType"]]

        # cypher = f"""
        # MATCH (a:Plan {{pmProductId: $src}}), (b:Plan {{pmProductId: $tgt}})
        # CREATE (a)-[r:{rel_type} {{
        #     changerule_base: $base,
        #     changerule_date: $date
        # }}]->(b)
        # """
        cypher = f"""
        MATCH (a:요금제 {{고유ID: $src}}), (b:요금제 {{고유ID: $tgt}})
        CREATE (a)-[r:{rel_type} {{
            변경조건: $base,
            변경일: $date
        }}]->(b)
        """
        session.run(cypher, src=src, tgt=tgt, base=base, date=date)

driver.close()

### benefit > data
- 데이터 충전, 선물 혜택 관련 정보
- Label: 테이블의 각 칼럼을 label로 분해 (그래프의 관점에서 이렇게 구성하는것이 더 타당해 보여서)
- Property
  - value
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Benefit') -[HAS_SubCategory]-> Category('Benefit_Data') -[CONTAINS]-> 칼럼명('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_칼럼명]-> 칼럼명('...')

In [31]:
meta_path = os.path.join(HOME_DIR, "benefit", "data", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [32]:
meta_table.head(5)

,pmProductId,datagiftreceivingavailability,maximumshareamount,datarefillamount,datarefillcoupongiftingavailability
0,PA00000001,True,2.0,15.0,True
1,PA00000002,True,2.0,1.8,True
2,PA00000003,True,2.0,5.0,True
3,PA00000004,True,2.0,30.0,True
4,PA00000005,True,2.0,54.0,True


In [33]:
cols = [
    "datagiftreceivingavailability",
    "maximumshareamount",
    "datarefillamount",
    "datarefillcoupongiftingavailability"
]

In [34]:
kor_cols = ["데이터선물받기가능여부", "최대데이터선물가능용량", "데이터리필가능용량", "데이터리필쿠폰선물가능여부"]
kor_cols_map = { x:y for x, y in zip(cols, kor_cols) }

In [35]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    # benefit_data 노드 하나 생성하고 Benefit 노드와 연결 (그냥 data로 하면 다른 레이블과 혼동됨)
    # session.run("MERGE (:Category {name:'Benefit_Data'})")
    # session.run("""
    #     MATCH (b:Category {name: 'Benefit'})
    #     MATCH (bd:Category {name:'Benefit_Data'})
    #     MERGE (b)-[:HAS_SubCategory]->(bd)
    # """)
    session.run("MERGE (:카테고리 {이름:'데이터_리필혜택'})")
    session.run("""
        MATCH (b:카테고리 {이름: '리필혜택'})
        MATCH (bd:카테고리 {이름:'데이터_리필혜택'})
        MERGE (b)-[:서브카테고리]->(bd)
    """)
driver.close()

In [36]:
driver = GraphDatabase.driver(uri, auth=(username, password))
with driver.session() as session:
    for col in cols:
        # 칼럼명을 레이블로 하고, 해당 칼럼의 유니크한 값들을 각각 노드로 생성
        for val in meta_table[col].dropna().unique():
            session.run(
                # f"MERGE (n:`{col}` {{ value: $val }})",
                f"MERGE (n:`{kor_cols_map[col]}` {{ 값: $val }})",
                val=val
            )

    # 상품과 관계 생성
    for record in meta_table.to_dict("records"):
        pm = record["pmProductId"]
        for col in cols:
            val = record[col]
            if pd.isna(val):
                continue

            # session.run(
            #     f"""
            #     MATCH (p:Plan {{pmProductId: $pm}})
            #     MATCH (v:`{col}`    {{ value: $val }})
            #     MERGE (p)-[:HAS_{col}]->(v)
            #     """,
            #     pm=pm, val=val
            # )
            session.run(
                f"""
                MATCH (p:요금제 {{고유ID: $pm}})
                MATCH (v:`{kor_cols_map[col]}` {{ 값: $val }})
                MERGE (p)-[:{kor_cols_map[col]}]->(v)
                """,
                pm=pm, val=val
            )

driver.close()

In [37]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:  
    
    for record in meta_table.to_dict("records"):
        for col in cols:
            val = record[col]
            if pd.isna(val):
                continue
            # session.run(
            #     f"""
            #     MATCH (b:Category {{name:'Benefit_Data'}})
            #     MATCH (n:{col})
            #     MERGE (b)-[:CONTAINS]->(n)
            #     """
            # )
            session.run(
                f"""
                MATCH (b:카테고리 {{이름:'데이터_리필혜택'}})
                MATCH (n:{kor_cols_map[col]})
                MERGE (b)-[:데이터_리필혜택_종류]->(n)
                """
            )
driver.close()

### benefit > product
- 미처리

### benefit > voice
- 통화 충전 관련 정보
- Label
  - voicecallrefill
- Property
  - refillamount
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Benefit') -[HAS_SubCategory]-> Category('Benefit_Voice') -[CONTAINS]-> voicecallrefill('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_voicecallrefill(voicecallrefillrange:[...])]-> 칼럼명('...')

In [38]:
meta_path = os.path.join(HOME_DIR, "benefit", "voice", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [39]:
meta_table["voicecallrefillrange"] = meta_table["voicecallrefillrange|range"].apply(lambda x: ast.literal_eval(x))

In [40]:
del meta_table["voicecallrefillrange|range"]

In [41]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    # benefit_voice 노드 하나 생성하고 Benefit 노드와 연결 (그냥 voice로 하면 다른 레이블과 혼동됨)
    # session.run("MERGE (:Category {name:'Benefit_Voice'})")
    # session.run("""
    #     MATCH (b:Category {name: 'Benefit'})
    #     MATCH (bv:Category {name:'Benefit_Voice'})
    #     MERGE (b)-[:HAS_SubCategory]->(bv)
    # """)
    session.run("MERGE (:카테고리 {이름:'음성_리필혜택'})")
    session.run("""
        MATCH (b:카테고리 {이름: '리필혜택'})
        MATCH (bv:카테고리 {이름:'음성_리필혜택'})
        MERGE (b)-[:서브카테고리]->(bv)
    """)
driver.close()

In [42]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for row in meta_table.to_dict("records"):
        pm_id   = row["pmProductId"]
        ranges  = row["voicecallrefillrange"]    # 리스트 타입
        refill  = float(row["refillamount"])     # float
    
        # session.run(
        #         """
        #         MATCH (p:Plan {pmProductId: $pm})
        #         MERGE (v:voicecallrefill {refillamount: $refill})
        #         CREATE (p)-[:HAS_voicecallrefill {
        #             voicecallrefillrange: $ranges
        #         }]->(v)
        #         """,
        #         pm=pm_id,
        #         refill=refill,
        #         ranges=ranges
        #     )
        session.run(
                """
                MATCH (p:요금제 {고유ID: $pm})
                MERGE (v:음성리필 {리필음성통화량: $refill})
                CREATE (p)-[:음성통화리필항목 {음성통화리필항목: $ranges}]->(v)
                """,
                pm=pm_id,
                refill=refill,
                ranges=ranges
            )

driver.close()

In [43]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:  
    # session.run(
    #     f"""
    #     MATCH (b:Category {{name:'Benefit_Voice'}})
    #     MATCH (n:voicecallrefill)
    #     MERGE (b)-[:CONTAINS]->(n)
    #     """
    # )
    session.run(
        f"""
        MATCH (b:카테고리 {{이름:'음성_리필혜택'}})
        MATCH (n:음성리필)
        MERGE (b)-[:음성_리필혜택_종류]->(n)
        """
    )
driver.close()

### campaign > targetproduct
- 각각의 캠페인을 별도의 노드로 생성
- Label
  - Campaign
- Property
  - productName, legacyProductId
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Campaign') -[HAS_Campaign]-> Campaign('...')

In [44]:
meta_path = os.path.join(HOME_DIR, "campaign", "targetproduct.csv")
meta_table = pd.read_csv(meta_path)

In [45]:
meta_table.head(5)

,pmProductId,productName,legacyProductId
0,BB00000001,T공시지원금,NaN
1,BA00000031,선택약정,NaN
2,BA00000028,0히어로 혜택,NaN
3,BA00000073,요금약정할인제도,NaN
4,BA00000047,Wavve 2천원 할인,NaN


In [46]:
records = meta_table.to_dict("records")

driver = GraphDatabase.driver(uri, auth=(username, password))
with driver.session() as session:
    # session.run(
    #     """
    #     UNWIND $rows AS row
    #     MERGE (c:Campaign {pmProductId: row.pmProductId})
    #     SET c.productName      = row.productName,
    #         c.legacyProductId = row.legacyProductId
    #     WITH c
    #     MATCH (category:Category {name: 'Campaign'})
    #     MERGE (category)-[:HAS_Campaign]->(c)
    #     """,
    #     rows=records
    # )
    session.run(
        """
        UNWIND $rows AS row
        MERGE (c:혜택 {고유ID: row.pmProductId})
        SET c.상품이름 = row.productName,
            c.과거상품ID = row.legacyProductId
        WITH c
        MATCH (category:카테고리 {이름: '혜택'})
        MERGE (category)-[:혜택상품]->(c)
        """,
        rows=records
    )
driver.close()

### campaign > relation
- 위의 캠페인을 요금제 상품과 연결
- relation의 property를 relationshipType으로 정의

In [47]:
meta_path = os.path.join(HOME_DIR, "campaign", "relation.csv")
meta_table = pd.read_csv(meta_path)

In [48]:
meta_table.head(5)

,pmProductId,targetProductId,relationshipType
0,PA00000001,BB00000001,signupPreTermination
1,PA00000001,BA00000031,signupConcurrentTermination
2,PA00000001,BA00000028,terminationPreTermination
3,PA00000001,BA00000073,terminationPreTermination
4,PA00000001,BA00000047,terminationConcurrentTermination


In [49]:
meta_table["relationshipType"].unique()

array(['signupPreTermination', 'signupConcurrentTermination',
       'terminationPreTermination', 'terminationConcurrentTermination',
       'signupConcurrentSignup'], dtype=object)

In [50]:
kor_relationshipType_map = {
    "signupPreTermination": "가입이전해지",
    "signupConcurrentTermination": "가입동시해지",
    "terminationPreTermination": "해지이전해지",
    "terminationConcurrentTermination": "해지동시해지",
    "signupConcurrentSignup": "가입동시가입",
}

meta_table["relationshipType"] = meta_table["relationshipType"].apply(lambda x: kor_relationshipType_map[x]) 

In [51]:
records = meta_table.to_dict("records")

In [52]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    # session.run(
    #     """
    #     UNWIND $rows AS row
    #     MATCH (p:Plan     {pmProductId: row.pmProductId})
    #     MATCH (c:Campaign {pmProductId: row.targetProductId})
    #     MERGE (p)-[r:HAS_signupcondition]->(c)
    #     SET r.relationshipType = row.relationshipType
    #     """,
    #     rows=records
    # )
    session.run(
        """
        UNWIND $rows AS row
        MATCH (p:요금제     {고유ID: row.pmProductId})
        MATCH (c:혜택 {고유ID: row.targetProductId})
        MERGE (p)-[r:가입해지조건]->(c)
        SET r.가입해지조건 = row.relationshipType
        """,
        rows=records
    )

driver.close()

### commonRule
- 미처리

### customerInfo
- 가입조건에 대한 테이블, 일단 이 중에서 agerule만 사용
- Label: MinAge, MaxAge
- Property
  - value
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('CustomerInfo') -[CONTAINS]-> MinAge('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_MinAge]-> MinAge('...')

In [53]:
meta_path = os.path.join(HOME_DIR, "customerinfo", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [54]:
meta_table.head(5)

,pmProductId,welfarediscounttypeavailability,agerule,businesscustomersubtyperule,customertyperule,individualcustomersubtyperule,organizationcustomersubtyperule
0,PA00000001,{},"[0, 150]","['SKTelecom㈜', 'SKBroadband㈜']",['개인'],"['공무원', '일반']",{}
1,PA00000002,{},"[0, 150]","['SKTelecom㈜', 'SKBroadband㈜']",['개인'],"['공무원', '일반']",{}
2,PA00000003,{},"[0, 150]","['SKTelecom㈜', 'SKBroadband㈜']",['개인'],"['공무원', '일반']",{}
3,PA00000004,{},"[0, 150]","['SKTelecom㈜', 'SKBroadband㈜']",['개인'],"['공무원', '일반']",{}
4,PA00000005,{},"[0, 150]","['SKTelecom㈜', 'SKBroadband㈜']",['개인'],"['공무원', '일반']",{}


In [55]:
meta_table["agerule"] = meta_table["agerule"].apply(lambda x: [ int(y) for y in ast.literal_eval(x) ])

In [56]:
meta_table.head(5)

,pmProductId,welfarediscounttypeavailability,agerule,businesscustomersubtyperule,customertyperule,individualcustomersubtyperule,organizationcustomersubtyperule
0,PA00000001,{},"[0, 150]","['SKTelecom㈜', 'SKBroadband㈜']",['개인'],"['공무원', '일반']",{}
1,PA00000002,{},"[0, 150]","['SKTelecom㈜', 'SKBroadband㈜']",['개인'],"['공무원', '일반']",{}
2,PA00000003,{},"[0, 150]","['SKTelecom㈜', 'SKBroadband㈜']",['개인'],"['공무원', '일반']",{}
3,PA00000004,{},"[0, 150]","['SKTelecom㈜', 'SKBroadband㈜']",['개인'],"['공무원', '일반']",{}
4,PA00000005,{},"[0, 150]","['SKTelecom㈜', 'SKBroadband㈜']",['개인'],"['공무원', '일반']",{}


In [57]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for row in meta_table.to_dict("records"):
        pm_id = row["pmProductId"]
        age_range = row["agerule"]
        min_age = min(age_range)
        max_age = max(age_range)
    
        # session.run(
        #         """
        #         MATCH (p:Plan {pmProductId: $pm})
        #         MATCH (c:Category {name: 'CustomerInfo'})
        #         MERGE (ma:MinAge {value: $min_age})
        #         MERGE (xa:MaxAge {value: $max_age})
        #         MERGE (p)-[:HAS_minAge]->(ma)
        #         MERGE (p)-[:HAS_maxAge]->(xa)
        #         MERGE (c)-[:CONTAINS]->(ma)
        #         MERGE (c)-[:CONTAINS]->(xa)
        #         """,
        #         pm=pm_id,
        #         min_age = min_age,
        #         max_age = max_age
        #     )
        session.run(
            """
            MATCH (p:요금제 {고유ID: $pm})
            MATCH (c:카테고리 {이름: '가입조건'})
            MERGE (ma:최소나이 {값: $min_age})
            MERGE (xa:최대나이 {값: $max_age})
            MERGE (p)-[:최소가입가능나이]->(ma)
            MERGE (p)-[:최대가입가능나이]->(xa)
            MERGE (c)-[:가입조건_최소나이]->(ma)
            MERGE (c)-[:가입조건_최대나이]->(xa)
            """,
            pm=pm_id,
            min_age = min_age,
            max_age = max_age
        )

driver.close()

### data
- 요금제 별 데이터 정책
- Label: 각 칼럼을 label로 하고, 각 칼럼의 유니크한 값들을 노드로 정의
- Property
  - value
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Data') -[Contains]-> 칼럼명('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_칼럼명]-> 칼럼명('...')

In [58]:
meta_path = os.path.join(HOME_DIR, "data", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [59]:
meta_table.head(5)

,pmProductId,includeddataforsharingandtethering,includeddataseparatesetting,includedmvoip,appliedspeed,generaldataexceedlimit|availabletoapply,includeddata,seniordataexceedlimit|availabletoapply
0,PA00000001,15.0,0,15.0,1024,False,15.0,False
1,PA00000002,1.8,0,1.8,0,True,1.8,False
2,PA00000003,5.0,0,5.0,1024,False,5.0,False
3,PA00000004,30.0,0,100.0,5120,False,100.0,False
4,PA00000005,54.0,0,200.0,5120,False,200.0,False


In [60]:
for col in meta_table.columns:
    print(col, meta_table[col].nunique())

pmProductId 125
includeddataforsharingandtethering 36
includeddataseparatesetting 1
includedmvoip 44
appliedspeed 5
generaldataexceedlimit|availabletoapply 2
includeddata 44
seniordataexceedlimit|availabletoapply 2


In [61]:
col_rename = {
    "generaldataexceedlimit|availabletoapply":  "generaldataexceedlimit",
    "seniordataexceedlimit|availabletoapply":  "seniordataexceedlimit"
}

In [62]:
meta_table = meta_table.rename(columns=col_rename)

In [63]:
meta_table.head(5)

,pmProductId,includeddataforsharingandtethering,includeddataseparatesetting,includedmvoip,appliedspeed,generaldataexceedlimit,includeddata,seniordataexceedlimit
0,PA00000001,15.0,0,15.0,1024,False,15.0,False
1,PA00000002,1.8,0,1.8,0,True,1.8,False
2,PA00000003,5.0,0,5.0,1024,False,5.0,False
3,PA00000004,30.0,0,100.0,5120,False,100.0,False
4,PA00000005,54.0,0,200.0,5120,False,200.0,False


In [64]:
cols = [
    "includeddataforsharingandtethering",
    # "includeddataseparatesetting",
    "includedmvoip",
    "appliedspeed",
    "generaldataexceedlimit",
    "includeddata",
    "seniordataexceedlimit"
]

In [65]:
kor_cols = [
    "기본제공데이터중공유가능용량",
    "기본제공데이터중mVoIP용량",
    "데이터소진후데이터제공속도",
    "데이터소진후최대금액및속도제한적용",
    "기본제공데이터용량",
    "시니어대상데이터소진후최대금액및속도제한적용"
]

In [66]:
kor_cols_map = { x:y for x, y in zip(cols, kor_cols) }

In [67]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for col in cols:
        # 해당 컬럼의 유니크 값만 뽑아서 노드 생성
        for val in meta_table[col].dropna().unique():
            session.run(
                f"""
                MATCH (c:카테고리 {{ 이름: '기본제공데이터' }})
                MERGE (n:`{kor_cols_map[col]}` {{ 값: $val }})
                MERGE (c)-[:기본제공테이터_종류]->(n)
                """,
                val=val
            )
    
    for record in meta_table.to_dict("records"):
        pm = record["pmProductId"]
        for col in cols:
            val = record[col]
            if pd.isna(val):
                continue
            # session.run(
            #     f"""
            #     MATCH (p:Plan {{pmProductId: $pm}})
            #     MATCH (v:`{col}` {{ value: $val }})
            #     MERGE (p)-[:HAS_{col}]->(v)
            #     """,
            #     pm=pm, val=val
            # )
            session.run(
                f"""
                MATCH (p:요금제 {{고유ID: $pm}})
                MATCH (v:`{kor_cols_map[col]}` {{ 값: $val }})
                MERGE (p)-[:{kor_cols_map[col]}]->(v)
                """,
                pm=pm, val=val
            )

driver.close()    
    

### optionData
- 미처리

### product > otherproduct
- 미처리 (비어있음)

### smstext
- Label: 각 칼럼
- Property
  - 각 칼럼의 유니크한 값
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('SmsText') -[Contains]-> 칼럼명('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_칼럼명]-> 칼럼명('...')

In [68]:
meta_path = os.path.join(HOME_DIR, "smstext", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [69]:
meta_table.head(5)

,pmProductId,textrange,includedtext
0,PA00000001,0,999999.0
1,PA00000002,0,999999.0
2,PA00000003,0,999999.0
3,PA00000004,0,999999.0
4,PA00000005,0,999999.0


In [70]:
cols = [ "textrange", "includedtext" ]
kor_cols = [ "기본제공문자별도설정", "문자기본제공량" ]
kor_cols_map = {x:y for x,y in zip(cols, kor_cols)}

In [71]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for col in cols:
        for val in meta_table[col].dropna().unique():
            # session.run(
            #     f"""
            #     MATCH (c:Category {{ name: 'SmsText' }})
            #     MERGE (n:{col} {{ value: $val }})
            #     MERGE (c)-[:CONTAINS]->(n)
            #     """,
            #     val=val
            # )
            session.run(
                f"""
                MATCH (c:카테고리 {{ 이름: '기본제공문자' }})
                MERGE (n:{kor_cols_map[col]} {{ 값: $val }})
                MERGE (c)-[:기본제공문자_종류]->(n)
                """,
                val=val
            )

    for record in meta_table.to_dict("records"):
        pm = record["pmProductId"]
        for col in cols:
            val = record[col]

            # session.run(
            #     f"""
            #     MATCH (p:Plan {{ pmProductId: $pm }}),
            #           (v:{col} {{ value: $val }})
            #     MERGE (p)-[:HAS_{col}]->(v)
            #     """,
            #     pm=pm, val=val
            # )
            session.run(
                f"""
                MATCH (p:요금제 {{ 고유ID: $pm }}),
                      (v:{kor_cols_map[col]} {{ 값: $val }})
                MERGE (p)-[:{kor_cols_map[col]}]->(v)
                """,
                pm=pm, val=val
            )

driver.close()

### topupinfo
- 미처리

### voice
- voicecallrange|providingamount와 includedvoicecall의 차이를 모르겠음
- includedvideoorvalueaddedcall, voicecallrange|providingamount, includedvoicecall만 사용
- Label: 각 칼럼
- Property
  - 각 칼럼의 유니크한 값
- Parent Relation
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Voice') -[Contains]-> 칼럼명('...')
  - DatabaseType('ProductMeta') -[HAS_Category]-> Category('Product') -[HAS_Plan]-> Plan('...') -[HAS_칼럼명]-> 칼럼명('...')

In [72]:
meta_path = os.path.join(HOME_DIR, "voice", "meta.csv")
meta_table = pd.read_csv(meta_path)

In [73]:
meta_table = meta_table.rename({
    "voicecallrange|providingamount": "voicecallrange_providingamount",
}, axis=1)

In [74]:
meta_table = meta_table[["pmProductId", "includedvideoorvalueaddedcall", "voicecallrange_providingamount", "includedvoicecall"]].copy()

In [75]:
meta_table.head(5)

,pmProductId,includedvideoorvalueaddedcall,voicecallrange_providingamount,includedvoicecall
0,PA00000001,300,0,999999
1,PA00000002,100,0,999999
2,PA00000003,300,0,999999
3,PA00000004,300,0,999999
4,PA00000005,300,0,999999


In [76]:
cols = [
    "includedvideoorvalueaddedcall",
    "voicecallrange_providingamount",
    "includedvoicecall"
]
kor_cols = [
    "기본제공부가및영상통화",
    "음성통화제공량",
    "기본제공음성통화량"
]
kor_cols_map = {x:y for x,y in zip(cols, kor_cols)}

In [77]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    for col in cols:
        for val in meta_table[col].dropna().unique():
            # session.run(
            #     f"""
            #     MATCH (c:Category {{ name: 'Voice' }})
            #     MERGE (n:`{col}` {{ value: $val }})
            #     MERGE (c)-[:CONTAINS]->(n)
            #     """,
            #     val=val
            # )
            session.run(
                f"""
                MATCH (c:카테고리 {{ 이름: '기본제공음성' }})
                MERGE (n:`{kor_cols_map[col]}` {{ 값: $val }})
                MERGE (c)-[:기본제공음성_종류]->(n)
                """,
                val=val
            )
            
    for record in meta_table.to_dict("records"):
        pm = record["pmProductId"]
        for col in cols:
            val = record[col]

            # session.run(
            #     f"""
            #     MATCH (p:Plan {{pmProductId: $pm}})
            #     MATCH (v:`{col}` {{value: $val}})
            #     MERGE (p)-[:HAS_{col}]->(v)
            #     """,
            #     pm=pm,
            #     val=val
            # )
            session.run(
                f"""
                MATCH (p:요금제 {{고유ID: $pm}})
                MATCH (v:`{kor_cols_map[col]}` {{값: $val}})
                MERGE (p)-[:{kor_cols_map[col]}]->(v)
                """,
                pm=pm,
                val=val
            )

driver.close()

### 모든 boolean type을 string type으로 변경
- langchain_neo4j의 enhanced_schema를 사용할 때 boolean인 property 하나만 있는 경우 에러나는 버그가 있음

In [78]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session() as session:
    session.run(
        f"""
        MATCH (n)
        UNWIND keys(n) AS key
        WITH n, key, n[key] AS value
        WHERE (value = true OR value = false)
        SET n[key] = CASE WHEN value = true THEN 'true' ELSE 'false' END
        """
    )

driver.close()